# Export Flipped ONNX

This notebook uses functions from `op_0103/export_flipped_onnx.py` directly.

In [8]:
from pathlib import Path
import os
import pandas as pd
import sys

REPO_ROOT = Path('/home/zx/Projects/DAC/STAFI')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from op_0103.export_flipped_onnx import (
    DEFAULT_OUT_DIR,
    DEFAULT_POLICY_ONNX,
    DEFAULT_VISION_ONNX,
    default_output_path,
    export_one,
    filter_specs,
    load_specs,
    metric_suffix,
    parse_timestamp,
)


In [9]:
# Config
plan_json = "/home/zx/Projects/DAC/STAFI/op_0103/out/important_bits_0103_taylor_seq100_b1_action_accel_top100_20260319-183756_run01_20260319-183800.json"
target_model = 'vision'   # 'vision', 'policy', or 'both'
source_key = 'plan'       # 'plan' or 'ranked'
topn = 2
strict = False

vision_onnx = Path(DEFAULT_VISION_ONNX)
policy_onnx = Path(DEFAULT_POLICY_ONNX)
out_dir = Path(DEFAULT_OUT_DIR)


In [10]:
# Load and preview the flip plan
payload, specs = load_specs(str(plan_json), source_key)
ts = parse_timestamp(str(plan_json), payload)
metric = metric_suffix(payload)
by_model = filter_specs(specs, target_model, topn)

print('plan_json =', plan_json)
print('timestamp =', ts)
print('metric =', metric)
print('models =', list(by_model.keys()))
for model_key, rows in by_model.items():
    print(f'  {model_key}: {len(rows)} flips')
    for row in rows[:5]:
        print('   ', row)

# Show the top-N bit flips as a table
preview_rows = []
for model_key, rows in by_model.items():
    for rank, row in enumerate(rows, start=1):
        preview_rows.append({
            'rank': rank,
            'model': row.model,
            'name': row.name,
            'flat': row.flat,
            'bit': row.bit,
        })

preview_df = pd.DataFrame(preview_rows)
preview_df

plan_json = /home/zx/Projects/DAC/STAFI/op_0103/out/important_bits_0103_taylor_seq100_b1_action_accel_top100_20260319-183756_run01_20260319-183800.json
timestamp = 20260319-183756
metric = action_desiredacceleration
models = ['vision']
  vision: 2 flips
    FlipSpec(model='vision', name='vision_model.vision._en.stem.0.reparam_conv.bias', flat=53, bit=13)
    FlipSpec(model='vision', name='vision_model.vision._en.stages.2.blocks.0.token_mixer.reparam_conv.weight', flat=598, bit=9)


,rank,model,name,flat,bit
0,1,vision,vision_model.vision._en.stem.0.reparam_conv.bias,53,13
1,2,vision,vision_model.vision._en.stages.2.blocks.0.toke...,598,9


In [11]:
# Export flipped ONNX
inputs = {
    'vision': os.path.abspath(str(vision_onnx)),
    'policy': os.path.abspath(str(policy_onnx)),
}

exported = {}
for model_key, model_specs in by_model.items():
    onnx_out = default_output_path(os.path.abspath(str(out_dir)), model_key, ts, metric, len(model_specs))
    export_one(
        model_key=model_key,
        onnx_in=inputs[model_key],
        onnx_out=onnx_out,
        specs=model_specs,
        strict=strict,
    )
    exported[model_key] = onnx_out

exported


[Done] target_model=vision
[Done] input=/home/zx/Projects/DAC/STAFI/op_0103/models/driving_vision.onnx
[Done] output=/home/zx/Projects/DAC/STAFI/op_0103/out/vision_model_20260319-183756_action_desiredacceleration_top2.onnx
[Done] requested=2 applied=2 skipped=0 non_finite_new=0
  idx=0 name=vision_model.vision._en.stem.0.reparam_conv.bias flat=53 bit=13 old=3.6230469 new=927.5 finite=True
  idx=1 name=vision_model.vision._en.stages.2.blocks.0.token_mixer.reparam_conv.weight flat=598 bit=9 old=1.4609375 new=1.9609375 finite=True


{'vision': '/home/zx/Projects/DAC/STAFI/op_0103/out/vision_model_20260319-183756_action_desiredacceleration_top2.onnx'}